# Memo Generation and Retrieval Evaluation

## Project context

This notebook extends the local AI Financial Research Agent prototype. It uses the Phase 0 chunk table and TF-IDF retrieval outputs to create a cited, template-based research memo. No paid APIs, OpenAI API calls, external LLMs, or online services are used.

## Phase 1 objective

Phase 1 adds four local capabilities:

- Retrieve evidence for finance research questions.
- Convert retrieved chunks into citation-style evidence references.
- Extract risk flags with transparent keyword rules.
- Evaluate retrieval coverage, source traceability, and memo grounding.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.config import CHUNKS_DIR, EVIDENCE_DIR, FIGURES_DIR, RETRIEVAL_DIR, REPORTS_DIR
from src.evaluation import (
    build_evaluation_summary,
    evaluate_answer_grounding,
    evaluate_retrieval_coverage,
    evaluate_source_traceability,
    extract_risk_flags,
)
from src.retrieval import create_cited_evidence_table, retrieve_evidence_for_questions
from src.visualization import (
    plot_evidence_count_by_question,
    plot_grounding_check_summary,
    plot_risk_flags_by_category,
    plot_source_traceability_status,
)

for directory in [EVIDENCE_DIR, RETRIEVAL_DIR, FIGURES_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

## Load chunks and evidence tables

The main Phase 1 input is `outputs/chunks/document_chunks.csv`, created in Phase 0. Existing evidence outputs are loaded for reference only; the notebook refreshes research question retrieval outputs below.

In [ ]:
chunks_path = CHUNKS_DIR / "document_chunks.csv"
phase0_evidence_path = EVIDENCE_DIR / "evidence_table.csv"
phase0_retrieval_path = RETRIEVAL_DIR / "sample_retrieval_results.csv"

chunks = pd.read_csv(chunks_path)
phase0_evidence = pd.read_csv(phase0_evidence_path) if phase0_evidence_path.exists() else pd.DataFrame()
phase0_retrieval = pd.read_csv(phase0_retrieval_path) if phase0_retrieval_path.exists() else pd.DataFrame()

print(f"Chunks loaded: {chunks.shape[0]} rows from {chunks['document_id'].nunique()} documents")
print(f"Phase 0 evidence rows available: {len(phase0_evidence)}")
chunks.head(3)

## Define research questions

These questions are designed to cover the core finance themes in the sample document set: lending risk, inflation and rates, payments adoption, profitability, portfolio risk, and management monitoring.

In [ ]:
research_questions = [
    "What are the main credit and lending risks?",
    "How do inflation and interest rates affect financial institutions?",
    "What does the corpus say about digital payments adoption?",
    "What affects bank profitability and funding costs?",
    "What market risks affect portfolio performance?",
    "What risks should management monitor?",
]
research_questions

## Retrieve evidence for memo sections

The notebook rebuilds a local TF-IDF index and retrieves the top five chunks for each research question.

In [ ]:
retrieval_results = retrieve_evidence_for_questions(chunks, research_questions, top_k=5)
cited_evidence = create_cited_evidence_table(retrieval_results)

retrieval_results.to_csv(RETRIEVAL_DIR / "research_question_retrieval.csv", index=False)
cited_evidence.to_csv(EVIDENCE_DIR / "cited_evidence_table.csv", index=False)

cited_evidence.head(10)

## Create cited evidence table

Citation IDs are compact references back to document chunks. They are not legal citations; they are traceability markers for this local portfolio prototype.

In [ ]:
cited_evidence[["query", "rank", "retrieval_score", "citation", "document_id", "title"]].head(12)

## Extract risk flags

Risk flags are extracted with keyword rules, not model-generated labels. This keeps the method transparent and easy to audit.

In [ ]:
risk_flags = extract_risk_flags(cited_evidence)
risk_flags.to_csv(EVIDENCE_DIR / "risk_flags.csv", index=False)

risk_summary = (
    risk_flags.groupby("risk_category", as_index=False)
    .size()
    .rename(columns={"size": "flag_count"})
    .sort_values("flag_count", ascending=False)
)
risk_summary

## Build template-based research memo

The memo uses fixed templates and selected retrieved evidence. This is intentionally not an LLM-generated memo. Each substantive section includes citations that can be checked against the evidence table.

In [ ]:
section_query_map = {
    "Credit and lending risk": "What are the main credit and lending risks?",
    "Inflation and rate sensitivity": "How do inflation and interest rates affect financial institutions?",
    "Digital payments and fintech adoption": "What does the corpus say about digital payments adoption?",
    "Bank profitability and funding cost": "What affects bank profitability and funding costs?",
    "Market and portfolio risk": "What market risks affect portfolio performance?",
    "Key risks to monitor": "What risks should management monitor?",
}


def top_evidence_for_query(query, n=2):
    subset = cited_evidence[cited_evidence["query"] == query].sort_values("rank").head(n)
    return subset.to_dict("records")


section_evidence_rows = []
section_citations = {}
for section, query in section_query_map.items():
    evidence_rows = top_evidence_for_query(query, n=2)
    section_citations[section] = [row["citation"] for row in evidence_rows]
    for row in evidence_rows:
        section_evidence_rows.append(
            {
                "section": section,
                "query": query,
                "citation": row["citation"],
                "document_id": row["document_id"],
                "chunk_id": row["chunk_id"],
                "retrieval_score": row["retrieval_score"],
            }
        )

summary_citations = []
for query in research_questions[:3]:
    summary_citations.extend([row["citation"] for row in top_evidence_for_query(query, n=1)])

top_categories = risk_summary.head(5)["risk_category"].tolist() if not risk_summary.empty else []

memo_sections = {
    "Executive summary": (
        "The sample corpus points to a finance research workflow where lending quality, macro sensitivity, "
        "payments adoption, profitability pressure, and portfolio volatility can be traced back to source chunks. "
        f"The strongest evidence areas in this run are supported by {', '.join(summary_citations)}. "
        "Because this is a local prototype, the memo should be read as a retrieval-grounded research draft rather than a final analyst report."
    ),
    "Credit and lending risk": (
        "The retrieved evidence highlights borrower selection, repayment behavior, delinquency movement, and early default signals as the core lending risks. "
        "For a fintech or bank portfolio, these are the areas where monitoring should connect application data, transaction behavior, and collections outcomes. "
        f"Supporting citations: {', '.join(section_citations['Credit and lending risk'])}."
    ),
    "Inflation and rate sensitivity": (
        "Inflation and interest-rate pressure appear in the corpus as funding-cost, affordability, loan-demand, and bond-valuation channels. "
        "The evidence suggests that higher rates can raise asset yields, but they can also pressure borrower repayment capacity and fixed-income portfolio values. "
        f"Supporting citations: {', '.join(section_citations['Inflation and rate sensitivity'])}."
    ),
    "Digital payments and fintech adoption": (
        "Digital payments adoption is linked to convenience, merchant acceptance, trust, reliability, and customer education. "
        "The retrieved evidence also shows that payments data can support credit scoring, merchant analytics, and liquidity insight, while fraud and dispute handling remain important operating risks. "
        f"Supporting citations: {', '.join(section_citations['Digital payments and fintech adoption'])}."
    ),
    "Bank profitability and funding cost": (
        "The corpus frames profitability as a balance between net interest margin, fee income, operating efficiency, credit costs, and funding mix. "
        "Rising policy rates can improve asset yields, but deposit repricing and wholesale funding costs can offset that benefit. "
        f"Supporting citations: {', '.join(section_citations['Bank profitability and funding cost'])}."
    ),
    "Market and portfolio risk": (
        "The retrieved market-risk evidence focuses on volatility, changing correlations, liquidity conditions, downside scenarios, and interest-rate shocks. "
        "For portfolio analysis, the useful control points are exposure concentration, drawdown behavior, duration sensitivity, and scenario testing. "
        f"Supporting citations: {', '.join(section_citations['Market and portfolio risk'])}."
    ),
    "Key risks to monitor": (
        "The rule-based risk flag scan most frequently surfaced "
        f"{', '.join(top_categories)}. "
        "These flags are useful as a first review queue, but a human analyst should validate whether each keyword match is material in context. "
        f"Supporting citations: {', '.join(section_citations['Key risks to monitor'])}."
    ),
    "Limitations": (
        "This memo is based on synthetic sample documents, TF-IDF retrieval, and deterministic templates. "
        "It does not use paid APIs, external LLMs, private company reports, live filings, or analyst judgment beyond the transparent rules in the notebook. "
        "The synthetic nature of the corpus is visible in the source document headers, so conclusions should be treated as workflow demonstration output. "
        f"Supporting citation: {summary_citations[0]}."
    ),
}

section_evidence_map = pd.DataFrame(section_evidence_rows)
section_evidence_map.to_csv(EVIDENCE_DIR / "memo_section_evidence_map.csv", index=False)
section_evidence_map.head(10)

In [ ]:
memo_lines = ["# Template-Based Financial Research Memo", ""]
for section, text in memo_sections.items():
    memo_lines.extend([f"## {section}", "", text, ""])

memo_text = "\n".join(memo_lines).strip() + "\n"
(REPORTS_DIR / "research_memo.md").write_text(memo_text, encoding="utf-8")
print(memo_text[:1200])

## Retrieval coverage evaluation

Coverage checks whether each research question returns evidence rows and whether those rows span at least one source document.

In [ ]:
coverage_df = evaluate_retrieval_coverage(cited_evidence, research_questions)
coverage_df.to_csv(RETRIEVAL_DIR / "coverage_by_question.csv", index=False)
coverage_df

## Source traceability evaluation

Traceability checks whether each retrieved row includes a citation, source document ID, title, chunk ID, and snippet.

In [ ]:
traceability_df = evaluate_source_traceability(cited_evidence)
traceability_df.to_csv(RETRIEVAL_DIR / "source_traceability.csv", index=False)
traceability_df.head(10)

## Grounding evaluation

Grounding checks whether memo citations appear in the evidence table. This is a basic citation consistency check, not a semantic truth test.

In [ ]:
grounding_df = evaluate_answer_grounding(memo_sections, cited_evidence)
grounding_df.to_csv(RETRIEVAL_DIR / "grounding_checks.csv", index=False)
grounding_df

## Evaluation summary

The evaluation summary combines coverage, traceability, and grounding checks into a compact artifact for Phase 2 polish.

In [ ]:
evaluation_summary = build_evaluation_summary(coverage_df, traceability_df, grounding_df)
evaluation_summary.to_csv(RETRIEVAL_DIR / "retrieval_evaluation_summary.csv", index=False)
evaluation_summary

## Figures

The charts summarize risk flags, evidence coverage, source traceability, and memo grounding.

In [ ]:
fig = plot_risk_flags_by_category(risk_flags)
fig.savefig(FIGURES_DIR / "risk_flags_by_category.png", dpi=160, bbox_inches="tight")

fig = plot_evidence_count_by_question(coverage_df)
fig.savefig(FIGURES_DIR / "evidence_count_by_question.png", dpi=160, bbox_inches="tight")

fig = plot_source_traceability_status(traceability_df)
fig.savefig(FIGURES_DIR / "source_traceability_status.png", dpi=160, bbox_inches="tight")

fig = plot_grounding_check_summary(grounding_df)
fig.savefig(FIGURES_DIR / "grounding_check_summary.png", dpi=160, bbox_inches="tight")

[
    FIGURES_DIR / "risk_flags_by_category.png",
    FIGURES_DIR / "evidence_count_by_question.png",
    FIGURES_DIR / "source_traceability_status.png",
    FIGURES_DIR / "grounding_check_summary.png",
]

## Limitations

- Documents are synthetic sample notes for a portfolio prototype.
- Retrieval is TF-IDF only and may miss semantic matches.
- Risk flags use keyword rules and can over-tag or under-tag evidence.
- Template-based memo generation improves reproducibility but does not replace analyst writing.
- Grounding checks verify citation presence, not full factual correctness.

## Next steps for Phase 2 polish

- Polish README and career-facing reports.
- Keep limitations visible and honest.
- Optionally add selected figure previews.
- Leave embeddings, paid APIs, and production document parsing as future improvements.